# RAG Retrieval Accuracy — Hybrid Facet Scoring (Method 3)

Replaces pure FAISS nearest-neighbour search with a **hybrid score**:

```
score(i) = beta * dense_sim(i) + gamma * facet_cosine(i, trait)
```

- `dense_sim` — converted from L2 distance of the dual-embedding index
- `facet_cosine` — cosine similarity between the query's 6 trait-relevant facets and the training essay's facet vector (from `facet_vectors.npy`)

**Why this helps:** the 768-d dense embedding clusters by topic/style; the 6-d facet cosine clusters by GPT-4o-mini's trait-specific signal. Combining them finds essays that are both stylistically and trait-similar.

**Index used:** `data/vector_db/essays_dual/` (same as `full_dual` baseline).

**Gamma sweep:** γ ∈ {0.0, 0.2, 0.4, 0.6, 0.8, 1.0} with β = 1 − γ. γ=0 reproduces the full_dual baseline.

**Prerequisites:**
- `data/vector_db/essays_dual/` — run `notebook/gpt/build_hybrid_index.ipynb`
- `data/profile_db/essays_val/` — run `rag_retrieve_accuracy_profile.ipynb`

In [1]:
import os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [2]:
VAL_CSV        = str(project_root / "data/split/essays/val.csv")
VAL_PROFILE_DB = str(project_root / "data/profile_db/essays_val")
DUAL_DB        = str(project_root / "data/vector_db/essays_dual")
RES_DIR        = str(project_root / "result/retrieve_accuracy/hybrid_facet")

K_VALUES     = [3, 5]
GAMMA_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]  # facet weight; beta = 1 - gamma

TRAITS = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

val_df = pd.read_csv(VAL_CSV)
print(f"Val split      : {len(val_df)} rows")

facet_npy = Path(DUAL_DB) / "facet_vectors.npy"
for label, p in [
    ("Dual DB",       DUAL_DB),
    ("Val profiles",  VAL_PROFILE_DB),
    ("facet_vectors", str(facet_npy)),
]:
    status = "OK" if Path(p).exists() else "MISSING"
    print(f"  {label:<16s}: [{status}]  {p}")

if not Path(DUAL_DB).exists():
    raise RuntimeError("Missing dual index — run build_hybrid_index.ipynb first.")
if not facet_npy.exists():
    raise RuntimeError("Missing facet_vectors.npy — rebuild dual index.")

Val split      : 247 rows
  Dual DB         : [OK]  F:\std\GR\code\model_x_ocean\data\vector_db\essays_dual
  Val profiles    : [OK]  F:\std\GR\code\model_x_ocean\data\profile_db\essays_val
  facet_vectors   : [OK]  F:\std\GR\code\model_x_ocean\data\vector_db\essays_dual\facet_vectors.npy


## Step 1 — Load index, facet matrix, and val profiles

In [3]:
from rag.faiss_index  import FAISSIndex
from rag.facet_vector import load_facet_matrix

dual_index = FAISSIndex(dimension=0)
dual_index.load(
    str(Path(DUAL_DB) / "vectors.faiss"),
    str(Path(DUAL_DB) / "vectors_meta.jsonl"),
)
N_TRAIN = dual_index._index.ntotal
print(f"Dual index: {N_TRAIN} vectors, dim={dual_index.dimension}")

# (N_TRAIN, 30) — one row per training essay, 30 facet values
facet_mat = load_facet_matrix(str(facet_npy))
print(f"Facet matrix: {facet_mat.shape}")

Dual index: 1974 vectors, dim=768
Facet matrix: (1974, 30)


In [4]:
from rag.profiler.store   import ProfileStore
from rag.profiler.prompts import FACETS

val_store_path = Path(VAL_PROFILE_DB) / "profile_store.jsonl"
if not val_store_path.exists():
    raise RuntimeError(
        f"Val profiles not found at {val_store_path}.\n"
        "Run rag_retrieve_accuracy_profile.ipynb to build them."
    )

val_store = ProfileStore(str(val_store_path))
val_store.load()
val_profiles = {
    int(e["user_id"].split("_")[1]): e
    for e in val_store.get_all() if e.get("valid")
}
print(f"Val profiles: {len(val_profiles)} / {len(val_df)}")

Val profiles: 247 / 247


## Step 2 — Compute val query embeddings and facet vectors

Uses the same dual embedding as the `full_dual` baseline (γ=0 should reproduce those numbers).

In [5]:
from rag.embedder     import _embed_single, get_embedding, ALPHA
from rag.facet_vector import facet_vector


def render_full_profile_text(entry: dict) -> str:
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines  = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)


def _fuse(raw_emb, prof_emb, alpha=ALPHA):
    fused = alpha * raw_emb + (1.0 - alpha) * prof_emb
    norm  = np.linalg.norm(fused)
    return (fused / norm) if norm > 0 else fused


N = len(val_df)

print(f"Encoding {N} raw posts ...")
t0 = time.time()
raw_embs = np.array(
    [_embed_single(str(row["text"])) for _, row in val_df.iterrows()],
    dtype="float32",
)
print(f"Done in {time.time()-t0:.1f}s  shape={raw_embs.shape}")

full_profile_texts = [
    render_full_profile_text(val_profiles[i]) if i in val_profiles else str(val_df.iloc[i]["text"])
    for i in range(N)
]
print(f"\nEncoding {N} full profiles ...")
t0 = time.time()
prof_embs = np.array(get_embedding(full_profile_texts), dtype="float32")
print(f"Done in {time.time()-t0:.1f}s")

# Dual query embeddings — identical to full_dual baseline
query_embs = np.array([_fuse(raw_embs[i], prof_embs[i]) for i in range(N)], dtype="float32")

# Query facet vectors (30-d)
query_fvecs = np.array(
    [facet_vector(val_profiles.get(i, {}), normalize=True) for i in range(N)],
    dtype="float32",
)
n_zero = int((np.linalg.norm(query_fvecs, axis=1) == 0).sum())
print(f"\nQuery facet vectors: {query_fvecs.shape}  (zero/missing: {n_zero})")

Encoding 247 raw posts ...
[embedder] Loading embedding model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>


Done in 564.1s  shape=(247, 768)

Encoding 247 full profiles ...
Done in 391.4s

Query facet vectors: (247, 30)  (zero/missing: 0)


## Step 3 — Pre-compute dense distances and per-trait facet similarities

Compute once, re-rank many times across (gamma, k, trait) combinations.

In [6]:
from rag.facet_vector import facet_cosine

# Dense: retrieve ALL N_TRAIN neighbours for every query (unsorted is fine; we re-rank)
print(f"FAISS search: {N} queries x {N_TRAIN} candidates ...")
t0 = time.time()
all_distances, all_indices = dual_index._index.search(query_embs, N_TRAIN)
# L2 distance -> similarity: higher is better
dense_sims_ranked = 1.0 / (1.0 + all_distances)   # (N, N_TRAIN), in FAISS-rank order
print(f"Done in {time.time()-t0:.1f}s")

# Rebuild dense_sims in index order (position 0..N_TRAIN-1) for hybrid scoring
dense_sims_index = np.empty((N, N_TRAIN), dtype="float32")
for i in range(N):
    dense_sims_index[i, all_indices[i]] = dense_sims_ranked[i]
print(f"Dense sims in index order: {dense_sims_index.shape}")

# Facet cosines per trait: mask to 6 trait-specific facets, then cosine
print("\nComputing trait-masked facet cosines ...")
t0 = time.time()
facet_sims = {}   # trait_code -> (N, N_TRAIN)
for trait_code in TRAITS:
    rows = [
        facet_cosine(query_fvecs[i], facet_mat, trait_code=trait_code)
        for i in range(N)
    ]
    facet_sims[trait_code] = np.stack(rows, axis=0).astype("float32")
    print(f"  {trait_code}: {facet_sims[trait_code].shape}")
print(f"Done in {time.time()-t0:.1f}s")

FAISS search: 247 queries x 1974 candidates ...
Done in 0.2s
Dense sims in index order: (247, 1974)

Computing trait-masked facet cosines ...
  cOPN: (247, 1974)
  cCON: (247, 1974)
  cEXT: (247, 1974)
  cAGR: (247, 1974)
  cNEU: (247, 1974)
Done in 0.2s


## Step 4 — Evaluate across (gamma, k, trait)

For each combination:
1. `score = (1-gamma)*dense_sim + gamma*facet_cosine` — both in index order
2. Rank all training essays by score descending
3. Filter to entries with a label for this trait, take top-k
4. Measure label match rate vs the query's true label

In [7]:
def normalize_label(val):
    s = str(val).strip().lower()
    if s in ("1", "1.0"): return "high"
    if s in ("0", "0.0"): return "low"
    return s


all_rows = []

for gamma in GAMMA_VALUES:
    beta = round(1.0 - gamma, 1)
    tag  = " <- dense only" if gamma == 0.0 else (" <- facet only" if gamma == 1.0 else "")
    print(f"\n{'='*60}")
    print(f"  gamma={gamma:.1f}  beta={beta:.1f}{tag}")
    print(f"{'='*60}")

    for k in K_VALUES:
        for trait_code, trait_full in TRAITS.items():
            f_sims = facet_sims[trait_code]           # (N, N_TRAIN)
            d_sims = dense_sims_index                 # (N, N_TRAIN)

            match_rates, hits = [], []

            for i, (_, row) in enumerate(val_df.iterrows()):
                true_label = normalize_label(row[trait_code])
                if true_label not in ("high", "low"):
                    continue

                scores      = beta * d_sims[i] + gamma * f_sims[i]   # (N_TRAIN,)
                ranked_idxs = np.argsort(scores)[::-1]

                filtered = []
                for idx in ranked_idxs:
                    if len(filtered) >= k:
                        break
                    meta = dual_index._meta[idx]
                    if trait_full in meta.get("trait_labels", {}):
                        filtered.append(meta)

                n       = len(filtered)
                matches = sum(
                    1 for r in filtered
                    if r["trait_labels"][trait_full] == true_label
                )
                match_rates.append(matches / n if n > 0 else 0.0)
                hits.append(int(matches > 0))

            mean_r = float(np.mean(match_rates))
            std_r  = float(np.std(match_rates))
            hit_r  = float(np.mean(hits))
            print(f"  k={k}  {trait_full:<30s}  match={mean_r:.4f}+/-{std_r:.4f}  hit={hit_r:.4f}")

            all_rows.append({
                "gamma":           gamma,
                "beta":            beta,
                "trait":           trait_full,
                "k":               k,
                "n_queries":       len(match_rates),
                "mean_match_rate": mean_r,
                "std_match_rate":  std_r,
                "hit_rate":        hit_r,
            })

summary_df = pd.DataFrame(all_rows)
print("\nEvaluation complete.")


  gamma=0.0  beta=1.0 <- dense only
  k=3  Openness to Experience          match=0.5169+/-0.2886  hit=0.8907
  k=3  Conscientiousness               match=0.5155+/-0.2902  hit=0.8826
  k=3  Extraversion                    match=0.5209+/-0.2804  hit=0.8988
  k=3  Agreeableness                   match=0.5358+/-0.2821  hit=0.9190
  k=3  Neuroticism                     match=0.5047+/-0.3144  hit=0.8462
  k=5  Openness to Experience          match=0.5174+/-0.2441  hit=0.9636
  k=5  Conscientiousness               match=0.5304+/-0.2270  hit=0.9717
  k=5  Extraversion                    match=0.5142+/-0.2285  hit=0.9636
  k=5  Agreeableness                   match=0.5401+/-0.2166  hit=0.9879
  k=5  Neuroticism                     match=0.5117+/-0.2554  hit=0.9433

  gamma=0.2  beta=0.8
  k=3  Openness to Experience          match=0.5466+/-0.4278  hit=0.6842
  k=3  Conscientiousness               match=0.5614+/-0.4021  hit=0.7490
  k=3  Extraversion                    match=0.5709+/-0.3868  hi

## Step 5 — Results

In [8]:
for k_val in K_VALUES:
    sub   = summary_df[summary_df["k"] == k_val]
    pivot = sub.pivot_table(index="trait", columns="gamma", values="mean_match_rate")
    if 0.0 in pivot.columns:
        for g in GAMMA_VALUES[1:]:
            if g in pivot.columns:
                pivot[f"delta_g{g:.1f}"] = (pivot[g] - pivot[0.0]).round(4)
    print(f"\n=== Mean match rate  (k={k_val}) ===")
    display(pivot.round(4))


=== Mean match rate  (k=3) ===


gamma,0.0,0.2,0.4,0.6,0.8,1.0,delta_g0.2,delta_g0.4,delta_g0.6,delta_g0.8,delta_g1.0
trait,,,,,,,,,,,
Agreeableness,0.5358,0.5722,0.5682,0.5547,0.5547,0.5412,0.0364,0.0324,0.0189,0.0189,0.0054
Conscientiousness,0.5155,0.5614,0.5574,0.5641,0.5722,0.5587,0.0459,0.0418,0.0486,0.0567,0.0432
Extraversion,0.5209,0.5709,0.5560,0.5547,0.5493,0.5385,0.0499,0.0351,0.0337,0.0283,0.0175
Neuroticism,0.5047,0.5735,0.5776,0.5749,0.5735,0.5965,0.0688,0.0729,0.0702,0.0688,0.0918
Openness to Experience,0.5169,0.5466,0.5547,0.5574,0.5695,0.5803,0.0297,0.0378,0.0405,0.0526,0.0634



=== Mean match rate  (k=5) ===


gamma,0.0,0.2,0.4,0.6,0.8,1.0,delta_g0.2,delta_g0.4,delta_g0.6,delta_g0.8,delta_g1.0
trait,,,,,,,,,,,
Agreeableness,0.5401,0.5619,0.5660,0.5652,0.5611,0.5474,0.0219,0.0259,0.0251,0.0211,0.0073
Conscientiousness,0.5304,0.5709,0.5619,0.5692,0.5700,0.5733,0.0405,0.0316,0.0389,0.0397,0.0429
Extraversion,0.5142,0.5652,0.5571,0.5595,0.5563,0.5506,0.0510,0.0429,0.0453,0.0421,0.0364
Neuroticism,0.5117,0.5668,0.5798,0.5806,0.5846,0.5854,0.0551,0.0680,0.0688,0.0729,0.0737
Openness to Experience,0.5174,0.5498,0.5530,0.5563,0.5660,0.5676,0.0324,0.0356,0.0389,0.0486,0.0502


In [9]:
for k_val in K_VALUES:
    sub   = summary_df[summary_df["k"] == k_val]
    pivot = sub.pivot_table(index="trait", columns="gamma", values="hit_rate")
    if 0.0 in pivot.columns:
        for g in GAMMA_VALUES[1:]:
            if g in pivot.columns:
                pivot[f"delta_g{g:.1f}"] = (pivot[g] - pivot[0.0]).round(4)
    print(f"\n=== Hit rate  (k={k_val}) ===")
    display(pivot.round(4))


=== Hit rate  (k=3) ===


gamma,0.0,0.2,0.4,0.6,0.8,1.0,delta_g0.2,delta_g0.4,delta_g0.6,delta_g0.8,delta_g1.0
trait,,,,,,,,,,,
Agreeableness,0.9190,0.7773,0.7490,0.7206,0.7328,0.7085,-0.1417,-0.1700,-0.1984,-0.1862,-0.2105
Conscientiousness,0.8826,0.7490,0.7449,0.7652,0.7692,0.7530,-0.1336,-0.1377,-0.1174,-0.1134,-0.1296
Extraversion,0.8988,0.7854,0.7895,0.7733,0.7692,0.7692,-0.1134,-0.1093,-0.1255,-0.1296,-0.1296
Neuroticism,0.8462,0.7571,0.7814,0.8219,0.8421,0.8543,-0.0891,-0.0648,-0.0243,-0.0040,0.0081
Openness to Experience,0.8907,0.6842,0.6680,0.6721,0.6883,0.6437,-0.2065,-0.2227,-0.2186,-0.2024,-0.2470



=== Hit rate  (k=5) ===


gamma,0.0,0.2,0.4,0.6,0.8,1.0,delta_g0.2,delta_g0.4,delta_g0.6,delta_g0.8,delta_g1.0
trait,,,,,,,,,,,
Agreeableness,0.9879,0.8259,0.8259,0.8138,0.8057,0.8097,-0.1619,-0.1619,-0.1741,-0.1822,-0.1781
Conscientiousness,0.9717,0.8178,0.8340,0.8340,0.8421,0.8907,-0.1538,-0.1377,-0.1377,-0.1296,-0.0810
Extraversion,0.9636,0.8583,0.8502,0.8381,0.8340,0.8381,-0.1053,-0.1134,-0.1255,-0.1296,-0.1255
Neuroticism,0.9433,0.8057,0.8219,0.8704,0.8826,0.8988,-0.1377,-0.1215,-0.0729,-0.0607,-0.0445
Openness to Experience,0.9636,0.7287,0.7166,0.7206,0.7287,0.6964,-0.2348,-0.2470,-0.2429,-0.2348,-0.2672


In [10]:
# Best gamma per (trait, k)
best = (
    summary_df
    .sort_values("mean_match_rate", ascending=False)
    .groupby(["trait", "k"], as_index=False)
    .first()[["trait", "k", "gamma", "mean_match_rate", "hit_rate"]]
)
print("=== Best gamma per (trait, k) ===")
display(best)

# Overall average across all traits
avg_by_gamma_k = (
    summary_df
    .groupby(["gamma", "k"])[["mean_match_rate", "hit_rate"]]
    .mean()
    .reset_index()
)
print("\n=== Average match rate across all traits per (gamma, k) ===")
display(avg_by_gamma_k.round(4))

=== Best gamma per (trait, k) ===


,trait,k,gamma,mean_match_rate,hit_rate
0,Agreeableness,3,0.2,0.572200,0.777328
1,Agreeableness,5,0.4,0.565992,0.825911
2,Conscientiousness,3,0.8,0.572200,0.769231
3,Conscientiousness,5,1.0,0.573279,0.890688
4,Extraversion,3,0.2,0.570850,0.785425
5,Extraversion,5,0.2,0.565182,0.858300
6,Neuroticism,3,1.0,0.596491,0.854251
7,Neuroticism,5,1.0,0.585425,0.898785
8,Openness to Experience,3,1.0,0.580297,0.643725
9,Openness to Experience,5,1.0,0.567611,0.696356



=== Average match rate across all traits per (gamma, k) ===


,gamma,k,mean_match_rate,hit_rate
0,0.0,3,0.5188,0.8874
1,0.0,5,0.5228,0.9660
2,0.2,3,0.5649,0.7506
3,0.2,5,0.5629,0.8073
4,0.4,3,0.5628,0.7466
5,0.4,5,0.5636,0.8097
6,0.6,3,0.5611,0.7506
7,0.6,5,0.5662,0.8154
8,0.8,3,0.5638,0.7603
9,0.8,5,0.5676,0.8186


## Step 6 — Save

In [11]:
os.makedirs(RES_DIR, exist_ok=True)

summary_path = os.path.join(RES_DIR, "accuracy_summary.csv")
best_path    = os.path.join(RES_DIR, "best_gamma.csv")

summary_df.to_csv(summary_path, index=False)
best.to_csv(best_path, index=False)

print(f"Saved summary  -> {summary_path}  ({len(summary_df)} rows)")
print(f"Saved best     -> {best_path}")

Saved summary  -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\hybrid_facet\accuracy_summary.csv  (60 rows)
Saved best     -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\hybrid_facet\best_gamma.csv
